In [0]:
catalog_name='products'

In [0]:
bronze_df=spark.table(f'{catalog_name}.bronze.brz_products')
bronze_df.columns

['id',
 'title',
 'description',
 'category',
 'price',
 'discountPercentage',
 'rating',
 'stock',
 'tags',
 'brand',
 'sku',
 'weight',
 'warrantyInformation',
 'shippingInformation',
 'availabilityStatus',
 'reviews',
 'returnPolicy',
 'minimumOrderQuantity',
 'images',
 'thumbnail',
 'dimensions.width',
 'dimensions.height',
 'dimensions.depth',
 'meta.createdAt',
 'meta.updatedAt',
 'meta.barcode',
 'meta.qrCode',
 'ingested_at',
 'data_source']

In [0]:
from pyspark.sql.functions import col
silver_products=bronze_df.select(
    col('id').cast('int').alias('product_id'),
    col('title'),
    col('description'),
    col('category'),
    col('price').cast('double'),
    col('discountPercentage').cast('double'),
    col('rating').cast('double'),
    col('stock').cast('int'),
    col('brand'),
    col('sku'),
    col('weight').cast('double'),
    col("warrantyInformation"),
    col("shippingInformation"),
    col('availabilityStatus'),
    col('returnPolicy'),
    col("minimumOrderQuantity").cast('int'),
    col('thumbnail'),
    #dim
    col("`dimensions.width`").cast('double').alias('dim_width'),
    col("`dimensions.height`").cast('double').alias('dim_height'),
    col("`dimensions.depth`").cast('double').alias('dim_depth'),
    #meta
    col("`meta.createdAt`").alias('created_at'),
    col("`meta.updatedAt`").alias('updated_at'),
    col("`meta.barcode`").alias('barcode'),
    col("`meta.qrCode`").alias('qr_code'),
    col('ingested_at'),
    col('data_source')
)
display(silver_products.limit(2))
display(silver_products.count())

product_id,title,description,category,price,discountPercentage,rating,stock,brand,sku,weight,warrantyInformation,shippingInformation,availabilityStatus,returnPolicy,minimumOrderQuantity,thumbnail,dim_width,dim_height,dim_depth,created_at,updated_at,barcode,qr_code,ingested_at,data_source
1,Essence Mascara Lash Princess,The Essence Mascara Lash Princess is a popular mascara known for its volumizing and lengthening effects. Achieve dramatic lashes with this long-lasting and cruelty-free formula.,beauty,9.99,10.48,2.56,99,Essence,BEA-ESS-ESS-001,4.0,1 week warranty,Ships in 3-5 business days,In Stock,No return policy,48,https://cdn.dummyjson.com/product-images/beauty/essence-mascara-lash-princess/thumbnail.webp,15.14,13.08,22.99,2025-04-30T09:41:02.053Z,2025-04-30T09:41:02.053Z,5784719087687,https://cdn.dummyjson.com/public/qr-code.png,2026-06-06T20:18:02.653Z,dummyjson_api
2,Eyeshadow Palette with Mirror,"The Eyeshadow Palette with Mirror offers a versatile range of eyeshadow shades for creating stunning eye looks. With a built-in mirror, it's convenient for on-the-go makeup application.",beauty,19.99,18.19,2.86,34,Glamour Beauty,BEA-GLA-EYE-002,9.0,1 year warranty,Ships in 2 weeks,In Stock,7 days return policy,20,https://cdn.dummyjson.com/product-images/beauty/eyeshadow-palette-with-mirror/thumbnail.webp,9.26,22.47,27.67,2025-04-30T09:41:02.053Z,2025-04-30T09:41:02.053Z,9170275171413,https://cdn.dummyjson.com/public/qr-code.png,2026-06-06T20:18:02.653Z,dummyjson_api


30

In [0]:
silver_products=silver_products.dropDuplicates(['product_id'])
print(silver_products.count())

30


In [0]:
silver_products=silver_products.filter(
    col('product_id').isNotNull()
)

silver_products=silver_products.filter(
    col('title').isNotNull()
)
silver_products.count()

30

In [0]:
silver_products.write.format('delta')\
    .mode('overwrite')\
    .option("overwriteSchema", "true")\
    .saveAsTable(f'{catalog_name}.silver.silver_products')

In [0]:
display(bronze_df.limit(5))

id,title,description,category,price,discountPercentage,rating,stock,tags,brand,sku,weight,warrantyInformation,shippingInformation,availabilityStatus,reviews,returnPolicy,minimumOrderQuantity,images,thumbnail,dimensions.width,dimensions.height,dimensions.depth,meta.createdAt,meta.updatedAt,meta.barcode,meta.qrCode,ingested_at,data_source
1,Essence Mascara Lash Princess,The Essence Mascara Lash Princess is a popular mascara known for its volumizing and lengthening effects. Achieve dramatic lashes with this long-lasting and cruelty-free formula.,beauty,9.99,10.48,2.56,99,"List(beauty, mascara)",Essence,BEA-ESS-ESS-001,4,1 week warranty,Ships in 3-5 business days,In Stock,"List(List(Would not recommend!, 2025-04-30T09:41:02.053Z, 3, eleanor.collins@x.dummyjson.com, Eleanor Collins), List(Very satisfied!, 2025-04-30T09:41:02.053Z, 4, lucas.gordon@x.dummyjson.com, Lucas Gordon), List(Highly impressed!, 2025-04-30T09:41:02.053Z, 5, eleanor.collins@x.dummyjson.com, Eleanor Collins))",No return policy,48,List(https://cdn.dummyjson.com/product-images/beauty/essence-mascara-lash-princess/1.webp),https://cdn.dummyjson.com/product-images/beauty/essence-mascara-lash-princess/thumbnail.webp,15.14,13.08,22.99,2025-04-30T09:41:02.053Z,2025-04-30T09:41:02.053Z,5784719087687,https://cdn.dummyjson.com/public/qr-code.png,2026-06-06T20:18:02.653Z,dummyjson_api
2,Eyeshadow Palette with Mirror,"The Eyeshadow Palette with Mirror offers a versatile range of eyeshadow shades for creating stunning eye looks. With a built-in mirror, it's convenient for on-the-go makeup application.",beauty,19.99,18.19,2.86,34,"List(beauty, eyeshadow)",Glamour Beauty,BEA-GLA-EYE-002,9,1 year warranty,Ships in 2 weeks,In Stock,"List(List(Great product!, 2025-04-30T09:41:02.053Z, 5, savannah.gomez@x.dummyjson.com, Savannah Gomez), List(Awesome product!, 2025-04-30T09:41:02.053Z, 4, christian.perez@x.dummyjson.com, Christian Perez), List(Poor quality!, 2025-04-30T09:41:02.053Z, 1, nicholas.bailey@x.dummyjson.com, Nicholas Bailey))",7 days return policy,20,List(https://cdn.dummyjson.com/product-images/beauty/eyeshadow-palette-with-mirror/1.webp),https://cdn.dummyjson.com/product-images/beauty/eyeshadow-palette-with-mirror/thumbnail.webp,9.26,22.47,27.67,2025-04-30T09:41:02.053Z,2025-04-30T09:41:02.053Z,9170275171413,https://cdn.dummyjson.com/public/qr-code.png,2026-06-06T20:18:02.653Z,dummyjson_api
3,Powder Canister,"The Powder Canister is a finely milled setting powder designed to set makeup and control shine. With a lightweight and translucent formula, it provides a smooth and matte finish.",beauty,14.99,9.84,4.64,89,"List(beauty, face powder)",Velvet Touch,BEA-VEL-POW-003,8,3 months warranty,Ships in 1-2 business days,In Stock,"List(List(Would buy again!, 2025-04-30T09:41:02.053Z, 4, alexander.jones@x.dummyjson.com, Alexander Jones), List(Highly impressed!, 2025-04-30T09:41:02.053Z, 5, elijah.cruz@x.dummyjson.com, Elijah Cruz), List(Very dissatisfied!, 2025-04-30T09:41:02.053Z, 1, avery.perez@x.dummyjson.com, Avery Perez))",No return policy,22,List(https://cdn.dummyjson.com/product-images/beauty/powder-canister/1.webp),https://cdn.dummyjson.com/product-images/beauty/powder-canister/thumbnail.webp,29.27,27.93,20.59,2025-04-30T09:41:02.053Z,2025-04-30T09:41:02.053Z,8418883906837,https://cdn.dummyjson.com/public/qr-code.png,2026-06-06T20:18:02.653Z,dummyjson_api
4,Red Lipstick,"The Red Lipstick is a classic and bold choice for adding a pop of color to your lips. With a creamy and pigmented formula, it provides a vibrant and long-lasting finish.",beauty,12.99,12.16,4.36,91,"List(beauty, lipstick)",Chic Cosmetics,BEA-CHI-LIP-004,1,3 year warranty,Ships in 1 week,In Stock,"List(List(Great product!, 2025-04-30T09:41:02.053Z, 4, liam.garcia@x.dummyjson.com, Liam Garcia), List(Great product!, 2025-04-30T09:41:02.053Z, 5, ruby.andrews@x.dummyjson.com, Ruby Andrews), List(Would buy again!, 2025-04-30T09:41:02.053Z, 5, clara.berry@x.dummyjson.com, Clara Berry))",7 days return policy,40

In [0]:
from pyspark.sql.functions import explode
silver_reviews=bronze_df.select(
    col('id').cast('int').alias('product_id'),
    explode('reviews').alias('review')
)
display(silver_reviews.limit(3))

product_id,review
1,"List(Would not recommend!, 2025-04-30T09:41:02.053Z, 3, eleanor.collins@x.dummyjson.com, Eleanor Collins)"
1,"List(Very satisfied!, 2025-04-30T09:41:02.053Z, 4, lucas.gordon@x.dummyjson.com, Lucas Gordon)"
1,"List(Highly impressed!, 2025-04-30T09:41:02.053Z, 5, eleanor.collins@x.dummyjson.com, Eleanor Collins)"


In [0]:
silver_reviews=silver_reviews.select(
    col('product_id'),
    col('review.rating').cast('double').alias('review_rate'),
    col('review.comment').alias('review_comment'),
    col('review.date').alias('review_date'),
    col('review.reviewername').alias('reviewer_name'),
    col('review.revieweremail').alias('reviewer_email')
)
display(silver_reviews)

product_id,review_rate,review_comment,review_date,reviewer_name,reviewer_email
1,3.0,Would not recommend!,2025-04-30T09:41:02.053Z,Eleanor Collins,eleanor.collins@x.dummyjson.com
1,4.0,Very satisfied!,2025-04-30T09:41:02.053Z,Lucas Gordon,lucas.gordon@x.dummyjson.com
1,5.0,Highly impressed!,2025-04-30T09:41:02.053Z,Eleanor Collins,eleanor.collins@x.dummyjson.com
2,5.0,Great product!,2025-04-30T09:41:02.053Z,Savannah Gomez,savannah.gomez@x.dummyjson.com
2,4.0,Awesome product!,2025-04-30T09:41:02.053Z,Christian Perez,christian.perez@x.dummyjson.com
2,1.0,Poor quality!,2025-04-30T09:41:02.053Z,Nicholas Bailey,nicholas.bailey@x.dummyjson.com
3,4.0,Would buy again!,2025-04-30T09:41:02.053Z,Alexander Jones,alexander.jones@x.dummyjson.com
3,5.0,Highly impressed!,2025-04-30T09:41:02.053Z,Elijah Cruz,elijah.cruz@x.dummyjson.com
3,1.0,Very dissatisfied!,2025-04-30T09:41:02.053Z,Avery Perez,avery.perez@x.dummyjson.com
4,4.0,Great product!,2025-04-30T09:41:02.053Z,Liam Garcia,liam.garcia@x.dummyjson.com


In [0]:
silver_reviews.write\
    .format('delta')\
    .mode('overwrite')\
    .option('overwriteSchema','true')\
    .saveAsTable(f'{catalog_name}.silver.silver_reviews')

In [0]:
silver_tags=bronze_df.select(
    col('id').cast('int').alias('product_id'),
    explode('tags').alias('tag')
)
display(silver_tags.limit(4))
print(silver_tags.count())

product_id,tag
1,beauty
1,mascara
2,beauty
2,eyeshadow


47


In [0]:
silver_tags.write\
    .format('delta')\
    .mode('overwrite')\
    .option('overwriteSchema','true')\
    .saveAsTable(f'{catalog_name}.silver.silver_tags')

In [0]:
display(spark.sql(
    """
    SELECT * 
    FROM products.silver.silver_products
    LIMIT 5
    """
))

product_id,title,description,category,price,discountPercentage,rating,stock,brand,sku,weight,warrantyInformation,shippingInformation,availabilityStatus,returnPolicy,minimumOrderQuantity,thumbnail,dim_width,dim_height,dim_depth,created_at,updated_at,barcode,qr_code,ingested_at,data_source
12,Annibale Colombo Sofa,"The Annibale Colombo Sofa is a sophisticated and comfortable seating option, featuring exquisite design and premium upholstery for your living room.",furniture,2499.99,14.4,3.92,60,Annibale Colombo,FUR-ANN-ANN-012,6.0,Lifetime warranty,Ships in 1 week,In Stock,7 days return policy,1,https://cdn.dummyjson.com/product-images/furniture/annibale-colombo-sofa/thumbnail.webp,12.75,20.55,19.06,2025-04-30T09:41:02.053Z,2025-04-30T09:41:02.053Z,1777662847736,https://cdn.dummyjson.com/public/qr-code.png,2026-06-06T20:18:02.653Z,dummyjson_api
18,Cat Food,Nutritious cat food formulated to meet the dietary needs of your feline friend.,groceries,8.99,9.58,3.13,46,null,GRO-BRD-FOO-018,10.0,1 year warranty,Ships overnight,In Stock,No return policy,18,https://cdn.dummyjson.com/product-images/groceries/cat-food/thumbnail.webp,18.08,9.26,21.86,2025-04-30T09:41:02.053Z,2025-04-30T09:41:02.053Z,1483991328610,https://cdn.dummyjson.com/public/qr-code.png,2026-06-06T20:18:02.653Z,dummyjson_api
16,Apple,"Fresh and crisp apples, perfect for snacking or incorporating into various recipes.",groceries,1.99,12.62,4.19,8,null,GRO-BRD-APP-016,9.0,3 year warranty,Ships in 2 weeks,In Stock,90 days return policy,7,https://cdn.dummyjson.com/product-images/groceries/apple/thumbnail.webp,13.66,11.01,9.73,2025-04-30T09:41:02.053Z,2025-04-30T09:41:02.053Z,7962803553314,https://cdn.dummyjson.com/public/qr-code.png,2026-06-06T20:18:02.653Z,dummyjson_api
5,Red Nail Polish,"The Red Nail Polish offers a rich and glossy red hue for vibrant and polished nails. With a quick-drying formula, it provides a salon-quality finish at home.",beauty,8.99,11.44,4.32,79,Nail Couture,BEA-NAI-NAI-005,8.0,1 month warranty,Ships overnight,In Stock,No return policy,22,https://cdn.dummyjson.com/product-images/beauty/red-nail-polish/thumbnail.webp,21.63,16.48,29.84,2025-04-30T09:41:02.053Z,2025-04-30T09:41:02.053Z,4063010628104,https://cdn.dummyjson.com/public/qr-code.png,2026-06-06T20:18:02.653Z,dummyjson_api
10,Gucci Bloom Eau de,"Gucci Bloom by Gucci is a floral and captivating fragrance, with notes of tuberose, jasmine, and Rangoon creeper. It's a modern and romantic scent.",fragrances,79.99,14.39,2.74,91,Gucci,FRA-GUC-GUC-010,7.0,6 months warranty,Ships overnight,In Stock,No return policy,2,https://cdn.dummyjson.com/product-images/fragrances/gucci-bloom-eau-de/thumbnail.webp,20.92,21.68,11.2,2025-04-30T09:41:02.053Z,2025-04-30T09:41:02.053Z,3170832177880,https://cdn.dummyjson.com/public/qr-code.png,2026-06-06T20:18:02.653Z,dummyjson_api
